# Step 1 — Data Loading & EDA

Load the medical symptom dataset from HuggingFace Hub and upload to S3.

In [ ]:
import subprocess
subprocess.run(["pip", "install", "datasets", "-q"])

In [ ]:
import boto3
import pandas as pd
from datasets import load_dataset

BUCKET = "YOUR-S3-BUCKET-NAME"
REGION = "us-east-2"
PREFIX = "symptom-data"

# Load directly from HuggingFace Hub
dataset = load_dataset("gretelai/symptom_to_diagnosis")

df_train = pd.DataFrame(dataset["train"])
df_test  = pd.DataFrame(dataset["test"])

print(f"Train size: {len(df_train)}")
print(f"Test size:  {len(df_test)}")
print(f"Columns: {df_train.columns.tolist()}")
print(df_train.head(3))

## EDA

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_train["output_text"].value_counts().plot(
    kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Disease Distribution (Training Set)")
axes[0].tick_params(axis="x", rotation=45)

df_train["input_text"].str.len().plot(
    kind="hist", ax=axes[1], bins=30, color="coral")
axes[1].set_title("Symptom Text Length Distribution")

plt.tight_layout()
plt.savefig("eda.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Number of classes: {df_train['output_text'].nunique()}")
print(f"Avg text length:   {df_train['input_text'].str.len().mean():.0f} chars")

## Upload to S3

In [ ]:
# Save locally then upload
df_train.to_csv("train.csv", index=False)
df_test.to_csv("test.csv",   index=False)

s3 = boto3.client("s3", region_name=REGION)
s3.upload_file("train.csv", BUCKET, f"{PREFIX}/raw/train.csv")
s3.upload_file("test.csv",  BUCKET, f"{PREFIX}/raw/test.csv")

# Verify
response = s3.list_objects_v2(Bucket=BUCKET, Prefix=f"{PREFIX}/raw/")
for obj in response["Contents"]:
    print(f"{obj['Key']}  ({obj['Size']} bytes)")
print("Upload complete ✅")